# 불확실성 보정 실습

**Calibration · 신뢰도 곡선 · Reliability Diagram**

모델이 제시한 확률이나 구간이 실제 빈도와 일치하는지 확인하고 맞추는 작업.

소재 분야에서 이해하기: 90% 구간에 실제로 90%가 들어가는지 검증 데이터로 확인한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 확률 보정 문서](https://scikit-learn.org/stable/modules/calibration.html)

## 1. 90% 구간에 정말 90%가 들어갈까

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from sklearn.ensemble import RandomForestRegressor

def data(n, seed):
    local = np.random.default_rng(seed)
    x = local.uniform(0, 1, (n, 2))
    y = np.sin(3 * x[:, 0]) + x[:, 1] ** 2 + local.normal(0, 0.15, n)
    return x, y

X_train, y_train = data(400, 0)
X_test, y_test = data(2000, 1)
forest = RandomForestRegressor(n_estimators=300, random_state=0).fit(X_train, y_train)
spread = np.array([tree.predict(X_test) for tree in forest.estimators_])
mean, std = spread.mean(0), spread.std(0)
print('앙상블 표준편차 평균 %.3f, 실제 잔차 표준편차 %.3f' % (std.mean(), (y_test - mean).std()))

In [ ]:
levels = np.linspace(0.05, 0.95, 19)
from scipy.stats import norm
observed = [np.mean(np.abs(y_test - mean) < norm.ppf(0.5 + level / 2) * std) for level in levels]
plt.plot(levels, observed, 'o-', label='raw ensemble spread')
plt.plot([0, 1], [0, 1], 'k--', label='perfectly calibrated')
plt.xlabel('nominal coverage'); plt.ylabel('observed coverage'); plt.legend(); plt.show()
print('곡선이 대각선 아래면 구간이 너무 좁아 과신하고 있다는 뜻입니다.')

## 2. 보정 계수 하나로 맞추기

검증 데이터에서 표준편차를 상수배 해 커버리지를 맞춥니다.

In [ ]:
X_valid, y_valid = data(1000, 2)
valid_spread = np.array([tree.predict(X_valid) for tree in forest.estimators_])
valid_mean, valid_std = valid_spread.mean(0), valid_spread.std(0)
scale = np.std((y_valid - valid_mean) / np.maximum(valid_std, 1e-9))
print('보정 계수 %.2f' % scale)

calibrated = [np.mean(np.abs(y_test - mean) < norm.ppf(0.5 + level / 2) * std * scale) for level in levels]
plt.plot(levels, observed, 'o-', label='before calibration')
plt.plot(levels, calibrated, 's-', label='after calibration')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('nominal coverage'); plt.ylabel('observed coverage'); plt.legend(); plt.show()
print('90%% 구간 실제 커버리지: 보정 전 %.3f -> 보정 후 %.3f'
      % (observed[levels.searchsorted(0.9)], calibrated[levels.searchsorted(0.9)]))

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#calibration)을 여세요.